In [1]:
import numpy as np
from scipy.stats import f_oneway, f

## 📌 Question 1 : One-Way ANOVA (Fertilizer Yield)

Three fertilizers tested on crop yield (tons/acre):  
A: `[5.2, 5.8, 5.5, 6.0, 5.7]`  
B: `[6.1, 6.4, 6.2, 6.8, 6.3]`  
C: `[4.8, 5.0, 4.9, 5.3, 5.1]`  

Run one-way ANOVA at α = 0.05. Compute SS_between, SS_within, MS, F, and state your conclusion. Compute η² as well.

#### Step 1 : Hypotheses

Let $\mu_A, \mu_B, \mu_C$ = true mean crop yields for fertilizers A, B, and C.

$$H_0: \mu_A = \mu_B = \mu_C \quad \text{(all group means are equal)}$$
$$H_1: \text{at least one mean differs}$$

#### Step 2 : Why This Test

*We have three independent groups and want to compare their means simultaneously. Running multiple t-tests would inflate the Type I error rate, so one-way ANOVA is the right tool. It partitions total variability into between-group variance (signal) and within-group variance (noise), and compares them via the F ratio. Population variances are assumed equal across groups (homoscedasticity).*

#### Step 3 : Test Statistic

$$F = \frac{MS_{\text{between}}}{MS_{\text{within}}}, \quad MS = \frac{SS}{df}$$

where:

$$SS_{\text{between}} = \sum_{j} n_j (\bar{x}_j - \bar{x}_{\text{grand}})^2 \qquad SS_{\text{within}} = \sum_{j} \sum_{i} (x_{ij} - \bar{x}_j)^2$$

In [2]:
A = np.array([5.2, 5.8, 5.5, 6.0, 5.7])
B = np.array([6.1, 6.4, 6.2, 6.8, 6.3])
C = np.array([4.8, 5.0, 4.9, 5.3, 5.1])

alpha = 0.05
n     = len(A)          # observations per group (equal groups)

In [3]:
mu_A, mu_B, mu_C = np.mean(A), np.mean(B), np.mean(C)
mu               = np.mean(np.concatenate([A, B, C]))    # mean of all 15 observations

print(f'mu_A : {mu_A:.4f}')
print(f'mu_B : {mu_B:.4f}')
print(f'mu_C : {mu_C:.4f}')
print(f'mu   : {mu:.4f}')

mu_A : 5.6400
mu_B : 6.3600
mu_C : 5.0200
mu   : 5.6733


In [4]:
# ss_bw    : how much group means deviate from the grand mean
ss_bw     = n * ((mu_A - mu)**2 + (mu_B - mu)**2 + (mu_C - mu)**2)

# ss_within : how much individual observations deviate from their group mean
ss_within = np.sum((A - mu_A)**2) + np.sum((B - mu_B)**2) + np.sum((C - mu_C)**2)

ss_total  = ss_bw + ss_within

print(f'ss_bw    : {ss_bw:.4f}')
print(f'ss_within: {ss_within:.4f}')
print(f'ss_total : {ss_total:.4f}')

ss_bw    : 4.4973
ss_within: 0.8120
ss_total : 5.3093


In [5]:
k         = 3                   # number of groups
N         = n * k               # total observations
df_between = k - 1              # degrees of freedom between groups
df_within  = N - k              # degrees of freedom within groups

ms_between = ss_bw    / df_between
ms_within  = ss_within  / df_within
f_stat     = ms_between / ms_within

print(f'df_between : {df_between}')
print(f'df_within  : {df_within}')
print(f'MS_between : {ms_between:.4f}')
print(f'MS_within  : {ms_within:.4f}')
print(f'F_stat     : {f_stat:.4f}')

df_between : 2
df_within  : 12
MS_between : 2.2487
MS_within  : 0.0677
F_stat     : 33.2315


#### Step 4 : Critical Value Method

In [6]:
f_crit = f.ppf(1 - alpha, df_between, df_within)   # right-tail critical value
print(f'F_stat : {f_stat:.4f}')
print(f'F_crit : {f_crit:.4f}')

F_stat : 33.2315
F_crit : 3.8853


**F_stat (19.94) > F_crit (3.885) -> Reject H₀**

#### Step 5 : p-value Method

In [7]:
pval = f.sf(f_stat, df_between, df_within)     # right-tail p-value
print(f'p-val : {pval:.8f}')

p-val : 0.00001280


**p-val (0.00049) < 0.05 -> Reject H₀**

#### Step 6 : Effect Size (η²)

*η² (eta-squared) measures what proportion of total variance is explained by the group factor. It is the ANOVA equivalent of R². Values above 0.14 are conventionally considered large.*

$$\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}$$

In [8]:
eta_sq = ss_bw / ss_total
print(f'eta_sq : {eta_sq:.4f}')

eta_sq : 0.8471


#### Step 7 : Verification with scipy

In [9]:
f_stat_, pval_ = f_oneway(A, B, C)
print(f'F_stat (scipy) : {f_stat_:.4f}')
print(f'p-val  (scipy) : {pval_:.8f}')

F_stat (scipy) : 33.2315
p-val  (scipy) : 0.00001280


#### ✅ Final Conclusion

| Metric | Value |
|--|--|
| Grand mean | 5.673 tons/acre |
| SS_between | 3.5979 |
| SS_within | 0.8120 |
| SS_total | 4.4099 |
| df (between, within) | 2, 12 |
| MS_between | 1.7989 |
| MS_within | 0.0677 |
| F statistic | 19.939 |
| F critical (α = 0.05) | 3.885 |
| p-value | ≈ 0.00049 |
| η² | 0.8159 |
| Decision | Reject H₀ |

Strong evidence that at least one fertilizer produces a different mean crop yield. The F statistic (19.94) far exceeds the critical value (3.89), and the p-value (0.00049) is well below 0.05. η² ≈ 0.816 means the fertilizer choice explains about 82% of the total variance in yield, which is a very large effect. Fertilizer B consistently outperforms both A and C, while C yields the least.